# Model Family Colab Runner

This notebook is a Colab wrapper around the existing repo scripts.
It keeps the original Qwen workflow intact and adds Llama-family runs as an extension.
You can run just the Qwen family, just the Llama family, or a combined sweep.


## Notes

- Recommended GPU: `T4`, `L4`, or `A100`.
- `meta-llama/Llama-3.1-8B-Instruct` style checkpoints may require Hugging Face access approval.
- `3a`, `3b`, and `3compare` resume from hacked checkpoints, which are currently Qwen-based in this repo.
- For family comparisons, the safest cross-family conditions are from-scratch runs like `0`, `1a/1b/1c`, `2a/2b`, `2c-*`, and `2d-*`.


In [ ]:
%%capture
import os
import subprocess

os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

!pip install --upgrade -qqq uv

try:
    import numpy, PIL
    _numpy = f"numpy=={numpy.__version__}"
    _pil = f"pillow=={PIL.__version__}"
except Exception:
    _numpy = "numpy"
    _pil = "pillow"

try:
    is_t4 = "Tesla T4" in subprocess.check_output(["nvidia-smi"]).decode()
except Exception:
    is_t4 = False

_vllm, _triton = ("vllm==0.9.2", "triton==3.2.0") if is_t4 else ("vllm==0.15.1", "triton")

!uv pip install -qqq --upgrade {_vllm} {_numpy} {_pil} torchvision bitsandbytes xformers unsloth
!uv pip install -qqq {_triton}
!uv pip install -qqq transformers==4.56.2 datasets peft accelerate numpy pillow openai anthropic python-dotenv huggingface_hub
!uv pip install -qqq --no-deps trl==0.22.2
!uv pip uninstall -y -qqq torchcodec || true


In [ ]:
from pathlib import Path

REPO_URL = "https://github.com/armaansandhu26/ippo.git"
REPO_DIR = Path("/content/ippo")

if not REPO_DIR.exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    print(f"Using existing repo at {REPO_DIR}")

%cd /content/ippo


In [ ]:
# Optional: needed for gated Llama repos.
from huggingface_hub import notebook_login

# notebook_login()


In [ ]:
from pathlib import Path

# Family presets. Qwen remains the existing baseline; Llama is additive.
MODEL_FAMILIES = {
    "qwen": [
        "Qwen/Qwen2.5-0.5B-Instruct",
        "Qwen/Qwen2.5-1.5B-Instruct",
        "Qwen/Qwen2.5-3B-Instruct",
        "Qwen/Qwen2.5-7B-Instruct",
    ],
    "llama": [
        "unsloth/Llama-3.2-1B-Instruct-bnb-4bit",
        "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
        "unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    ],
}

# Choose one of: "qwen", "llama", "all", or "custom"
MODEL_SCOPE = "llama"

# Used only when MODEL_SCOPE == "custom"
CUSTOM_MODELS = [
    "unsloth/Llama-3.2-3B-Instruct-bnb-4bit",
]

# If True, the notebook runs one command per model in the selected scope.
# If False, it runs only BASE_MODEL.
RUN_MODEL_SWEEP = True

# Single-model fallback when RUN_MODEL_SWEEP is False.
BASE_MODEL = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"

# Conditions from the scripts.
CONDITION = "2d-nonblind"
RUN_COMPARISON_SCRIPT = False

# You can use a single seed or a seed sweep.
SEEDS = [42]
BETA = 0.1
CACHE_DIR = "/content/hf-cache"
OUTPUT_BASE = Path("/content/outputs")

def resolve_models(scope: str) -> list[str]:
    if scope == "qwen":
        return MODEL_FAMILIES["qwen"]
    if scope == "llama":
        return MODEL_FAMILIES["llama"]
    if scope == "all":
        return MODEL_FAMILIES["qwen"] + MODEL_FAMILIES["llama"]
    if scope == "custom":
        return CUSTOM_MODELS
    raise ValueError(f"Unknown MODEL_SCOPE: {scope}")

SELECTED_MODELS = resolve_models(MODEL_SCOPE) if RUN_MODEL_SWEEP else [BASE_MODEL]

print({
    "model_scope": MODEL_SCOPE,
    "selected_models": SELECTED_MODELS,
    "condition": CONDITION,
    "comparison_script": RUN_COMPARISON_SCRIPT,
    "seeds": SEEDS,
    "beta": BETA,
})


In [ ]:
# Optional sanity-check: comment this cell out if you want to save time.
from unsloth import FastLanguageModel

model_to_check = SELECTED_MODELS[0]
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_to_check,
    max_seq_length=1024,
    load_in_4bit=True,
    cache_dir=CACHE_DIR,
)
print("Loaded successfully:", model_to_check)
del model


In [ ]:
import json
import shlex
import subprocess
from pathlib import Path

script = (
    "scripts/curriculum_hacked/train_time_prompt_opt_comparison.py"
    if RUN_COMPARISON_SCRIPT
    else "scripts/curriculum_hacked/train_time_prompt_opt.py"
)

run_plan = []
for model_name in SELECTED_MODELS:
    family = "qwen" if model_name.startswith("Qwen/") else "llama"
    model_slug = model_name.split("/")[-1]
    for seed in SEEDS:
        output_root = OUTPUT_BASE / family / CONDITION / model_slug / f"seed_{seed}"
        output_root.mkdir(parents=True, exist_ok=True)
        cmd = [
            "python3",
            script,
            "--condition", CONDITION,
            "--base-model", model_name,
            "--seed", str(seed),
            "--beta", str(BETA),
            "--cache-dir", CACHE_DIR,
            "--output-root", str(output_root),
        ]
        run_plan.append({
            "family": family,
            "model": model_name,
            "seed": seed,
            "output_root": str(output_root),
            "cmd": cmd,
        })

print(f"Prepared {len(run_plan)} runs")
print(json.dumps([{k: v for k, v in item.items() if k != 'cmd'} for item in run_plan], indent=2))


In [ ]:
# Execute the run plan sequentially.
for item in run_plan:
    print("\nRunning:")
    print(" ".join(shlex.quote(part) for part in item["cmd"]))
    subprocess.run(item["cmd"], check=True)


In [ ]:
# Summarize whatever completed.
import json
from pathlib import Path

summary_rows = []
for item in run_plan:
    final_eval = Path(item["output_root"]) / "final_eval.json"
    row = {
        "family": item["family"],
        "model": item["model"],
        "seed": item["seed"],
        "output_root": item["output_root"],
        "has_final_eval": final_eval.exists(),
    }
    if final_eval.exists():
        data = json.loads(final_eval.read_text())
        metrics = data.get("final_eval_metrics", {})
        row.update({
            "accuracy": metrics.get("accuracy"),
            "not_a_accuracy": metrics.get("not_a_accuracy"),
            "a_rate": metrics.get("a_rate"),
        })
    summary_rows.append(row)

print(json.dumps(summary_rows, indent=2))


## Example Configurations

- Existing Qwen baseline only:
  - `MODEL_SCOPE = "qwen"`
- Llama-only family sweep:
  - `MODEL_SCOPE = "llama"`
- Cross-family sweep in one notebook:
  - `MODEL_SCOPE = "all"`
- One custom shortlist:
  - `MODEL_SCOPE = "custom"`
  - edit `CUSTOM_MODELS`

That keeps Llama as an add-on while preserving the original Qwen path.
